In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Qdrant
from qdrant_client import QdrantClient
from langchain_ollama import OllamaEmbeddings
from qdrant_client.models import Distance, VectorParams


In [8]:

pdf_path = "/home/user/Documents/VSCode/CSN-RAG/operational_manual_XYZ_company.pdf"
loader = PyPDFLoader(pdf_path)
data = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(data)

embeddings = OllamaEmbeddings(model="nomic-embed-text")

client = QdrantClient(url="http://localhost:6333")

collection_name = "RAG-Project-Ollama"

''' 
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=768, distance=Distance.COSINE),
)

vector_store = Qdrant(
    client=client, 
    collection_name=collection_name, 
    embeddings=embeddings
)

vector_store.add_documents(chunks)
'''


print(f"Success! {len(chunks)} chunks stored in Qdrant.")


        

Success! 27 chunks stored in Qdrant.


In [2]:
from typing import TypedDict, Annotated
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage


In [3]:
class mystate(TypedDict):
    question: str
    answer: str
    context: str
    history: Annotated[list, add_messages]

In [4]:
def ret(state: mystate):
    query = state["question"]

    query_vector = embeddings.embed_query(query)
    
    response = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=3,
        with_payload=True
    )

    context = []
    for res in response.points:
        content = res.payload.get('page_content', "")
        
        score = res.score
        metadata = res.payload.get('metadata', {})
        page_no = metadata.get('page', 'unknown')
        formatted_chunk = f"{content}, score:{score:.3f}, page{page_no}"
        context.append(formatted_chunk)

    context_txt = "\n\n".join(context)
    
    return {"context": context_txt}

In [5]:
def gen(state: mystate):
    llm = ChatOllama(model="llama3.2", temperature=0)

    memory = state.get("history", [])[-8:]

    myprompt = f"""You are a strict chat assistant. 
    Analyze the provided context and answer the user question ONLY using the provided context or your conversation history.
    
    STRICT FORMAT RULES:
    1. Answer ONLY based on the context provided.
    2. Provide  Score: [Score], Page: [Page No] for every new response"
    3. If not in context, reply EXACTLY: "This information is not present in the provided document."
    4. Do not invent any facts.
    5. if the user asks about a previous answer or a citation for it, use your conversation history to provide the source.
    6. you can refer to the conversation history to understand references like 'it', 'last question', or 'that policy'

    Context:
    {state['context']}"""

    input_message = [SystemMessage(content=myprompt)] + memory + [HumanMessage(content=state["question"])] 

    response = llm.invoke(input_message)
    
    
    return {
        "answer": response.content,
        "history": [
            HumanMessage(content=state["question"]), 
            AIMessage(content=response.content)
        ]
    }

In [6]:
pipe = StateGraph(mystate)

pipe.add_node("ret_node", ret)
pipe.add_node("gen_node", gen)

pipe.add_edge("ret_node", "gen_node")
pipe.add_edge("gen_node", END)

pipe.set_entry_point("ret_node")

sys = pipe.compile()

In [57]:
test = "what is the leave policy?"

res = sys.invoke({"question": test})

print(res["answer"])

The leave policy includes the following entitlements:

* Annual Leave: 20 days, with unused leave expiring on March 31st of the following year.
* Sick Leave: 12 days, with mental health days categorized under sick leave and no explanation required for a 1-day absence.
* Maternity/Paternity: Primary caregivers receive 100% pay for 16 weeks, while secondary caregivers receive 100% pay for 6 weeks.

Source: Page 3


In [9]:
if __name__ == "__main__":
    
    chat_history=[]
    
    while True:
        user_input = input("you: ")
        if user_input.lower() == "exit": break
        result = sys.invoke({"question": user_input, "history": chat_history})
        chat_history = result.get("history", [])

        print("Full Result Keys:", result.keys())
        print(len(chat_history))
        print("you:", user_input)
        print("Bot:", result["answer"])



Full Result Keys: dict_keys(['question', 'answer', 'context', 'history'])
2
you: inclusive environment?
Bot: To ensure a psychologically safe environment, we adhere to the Inclusive Language Guide v4.0. 

●  Pronoun Respect: We respect individual identity. Employees are encouraged, though not required, to list their pronouns in Slack and Email signatures.

●  Neutral Terminology: Avoid gendered idioms. Use "Primary/Secondary" instead of terms that may be perceived as gendered.
Full Result Keys: dict_keys(['question', 'answer', 'context', 'history'])
4
you: do they provide parental leave facilty?
Bot: Score: 0.455, Page: 9

Yes, they provide parental leave facility. According to the provided context, 

●  Maternity/Paternity: Primary caregivers receive 100% pay for 16 weeks. Secondary caregivers receive 100% pay for 6 weeks.
